In [1]:
import requests
import json
import pandas as pd


API Call and JSON Parser

In [2]:
offset = 0 
base_url = "https://api.gbif.org/v1/occurrence/search"
payload = {"scientificName":"Cervus canadensis","country":"US","limit":300,'orderBy':'eventDate','order':'desc','offset':offset}
endofRecords = False
cleaned_list = []

while endofRecords is False and offset < 2000:

    ## make API call
    r= requests.get(base_url,params=payload)
    ##print(f"Encoded URL: {r.url}") 
    ##print(f"JSON Response: {r.json()}")
    results = r.json()['results']
    


    ## process results
    for row in results:
        filtered_row = {key: value for key, value in row.items() if key in (('gbifID','occurrenceID','decimalLatitude','decimalLongitude','stateProvince','country','eventDate','year','month','day','coordinateUncertaintyInMeters','recordedBy','institutionCode','gadm'))}
        filtered_row['county'] = filtered_row.get('gadm',{}).get('level2',{}).get('name')
        filtered_row['state'] = filtered_row.get('gadm',{}).get('level1',{}).get('name')
        filtered_row['country'] = filtered_row.get('gadm',{}).get('level0',{}).get('name')
        filtered_row.pop('gadm', None)
        filtered_row.pop('recordedBy', None)

        cleaned_list.append(filtered_row)

    ## increment offset
    offset = offset + 300
    payload['offset'] = offset
    
    ## update end of records
    endofRecords = r.json()['endOfRecords']

BigQuery Load

In [3]:
from google.cloud import bigquery
import os
import pandas as pd

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = r"C:\Users\lukej\Projects\wildlife-lakehouse\config\wildlife-lakehouse-8025d912066e.json"

client = bigquery.Client(project="wildlife-lakehouse")



In [5]:
## Create Table in Correct Project/Dataset
project_id = "wildlife-lakehouse"
dataset_id = 'wildlife_raw'
table_id = 'elk_sighting_raw'

table = f"{project_id}.{dataset_id}.{table_id}"

## Define Schema
schema = [
    bigquery.SchemaField("gbifID", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("eventDate", "TIMESTAMP", mode="NULLABLE"),
    bigquery.SchemaField("year", "INT64", mode="NULLABLE"),
    bigquery.SchemaField("month", "INT64", mode="NULLABLE"),
    bigquery.SchemaField("day", "INT64", mode="NULLABLE"),
    bigquery.SchemaField("county", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("state", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("stateProvince", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("country", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("decimalLatitude", "FLOAT64", mode="NULLABLE"),
    bigquery.SchemaField("decimalLongitude", "FLOAT64", mode="NULLABLE"),
    bigquery.SchemaField("occurrenceID", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("institutionCode", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("coordinateUncertaintyInMeters", "FLOAT64", mode="NULLABLE")
]

## Create Job Config
job_config = bigquery.LoadJobConfig(
    schema=schema,
    write_disposition="WRITE_TRUNCATE"
)

## Create Dataframe
df = pd.DataFrame(cleaned_list)

## Fix Data Type Issue of Event Date column not being in UTC before Load
df['eventDate'] = pd.to_datetime(df['eventDate'], utc=True, errors='coerce')

## Create Job
job = client.load_table_from_dataframe(df,table,job_config=job_config)

job.result()

print(f"Loaded {job.output_rows} rows into {table_id}.")

Loaded 2100 rows into elk_sighting_raw.


In [7]:
job = client.load_table_from_dataframe(df,table,job_config=job_config)

job.result()

print(f"Loaded {job.output_rows} rows into {table_id}.")

Loaded 2100 rows into elk_sighting_raw.
